# MolForge GRPO Training Pipeline
This notebook implements the Reinforcement Learning (GRPO) training pipeline for the MolForge environment.
We train the model using a **Proposer-Critic-Selector** architecture and targeted **reward shaping** to overcome local minima.

In [ ]:
!pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -U "trl>=0.21.0" peft accelerate bitsandbytes datasets matplotlib pandas huggingface_hub "openenv-core[core]>=0.2.3" rdkit jmespath xformers

In [ ]:
import os
import sys
from pathlib import Path

# Clone the repository
if not Path("/content/molt_lab").exists():
    !git clone https://github.com/Adhitya-Vardhan/molt_lab.git /content/molt_lab

# Add project root to path
if "/content/molt_lab" not in sys.path:
    sys.path.insert(0, "/content/molt_lab")
    
# Change working directory
os.chdir("/content/molt_lab")

In [ ]:
import time
import os

# Training Configuration
os.environ["MOLFORGE_REWARD_MODE"] = "curriculum"
os.environ["MOLFORGE_TRAINING_RANDOMIZATION"] = "1"

RL_MAX_STEPS = 80
NUM_GENERATIONS = 2
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-6
MAX_SEQ_LENGTH = 2048
MAX_PROMPT_LENGTH = 1536
MAX_COMPLETION_LENGTH = 384

RUN_NAME = time.strftime("molforge_grpo_%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(f"/content/molforge_rl_runs/{RUN_NAME}")
ADAPTER_SAVE_DIR = OUTPUT_DIR / "adapters"
PLOT_DIR = OUTPUT_DIR / "plots"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

### Reward Function & OpenEnv Integration
We implement a custom reward function that wraps the native `MolForgeEnvironment`. 
To prevent "reward hacking" (where the model endlessly farms `run_assay` for safe points), we apply targeted reward shaping.

In [ ]:
import json
from typing import Any, Dict, Tuple
from inference_common import (
    MolForgeAction,
    attach_reasoning_fields,
    attach_team_messages,
    extract_json,
)
from server.molforge_environment import MolForgeEnvironment
from models import MolForgeState

def replay_to_state(record: dict[str, Any]) -> MolForgeEnvironment:
    env = MolForgeEnvironment()
    env._state = MolForgeState(**record["state"])
    env._molecule = dict(record["molecule"])
    env._scenario = [s for s in env.SCENARIOS if s.scenario_id == env._state.scenario_id][0]
    return env

def evaluate_completion(prompt_str: str, completion_str: str, record: dict[str, Any]) -> Tuple[float, dict]:
    diagnostics = {"valid_json": False}
    try:
        action_dict = extract_json(completion_str)
        action = MolForgeAction(**action_dict)
    except Exception:
        return -1.2, diagnostics

    diagnostics["valid_json"] = True
    env = replay_to_state(record)
    
    # Create empty observation and attach reasoning
    observation = env._build_observation(reward=0.0, done=False, reward_components=[])
    action = attach_team_messages(observation, attach_reasoning_fields(observation, action))
    
    # Step the OpenEnv environment
    next_observation = env.step(action)
    reward = float(next_observation.reward)
    grader_scores = next_observation.metadata.get("terminal_grader_scores", {})
    
    # --- ANTI-REWARD-HACKING SHAPING ---
    if action.action_type == "run_assay" and reward > 0:
        reward *= 0.25  # Nerf assay farming
    elif action.action_type == "submit":
        sub_score = float(grader_scores.get("submission_score", 0.0))
        if sub_score > 0.0:
            reward += sub_score * 3.0  # Massive multiplier for submissions
    elif action.action_type == "edit" and reward > 0:
        reward *= 1.5  # Boost edits

    diagnostics.update({
        "action_type": action.action_type,
        "reward": reward,
        "done": next_observation.done,
    })
    return reward, diagnostics

def molforge_reward_func(prompts, completions, **kwargs) -> list[float]:
    rewards = []
    dataset_records = kwargs.get("record", [])
    
    for prompt_list, completion, record in zip(prompts, completions, dataset_records):
        prompt_str = prompt_list[-1]["content"] if isinstance(prompt_list, list) else str(prompt_list)
        completion_str = completion[0]["content"] if isinstance(completion, list) else str(completion)
        reward, _ = evaluate_completion(prompt_str, completion_str, record)
        rewards.append(reward)
    return rewards

### Model & Tokenizer Loading

In [ ]:
from unsloth import FastLanguageModel

# Set this to your SFT checkpoint
# You can set this to a local path or a Hugging Face repo
SFT_ADAPTER_PATH = "/content/drive/MyDrive/Qwen_3.5_finetune/qwen3_5_2b_lora_adapters_compact_v4" # <-- Change to your path

print("Loading model and applying Unsloth optimizations...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Enable fast training paths
FastLanguageModel.for_training(model)

# Extract underlying tokenizer if it is wrapped in a vision processor
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer

### GRPO Training Loop

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
from scripts.generate_sft_compact_policy_v4_dataset import compact_action_payload, COMPACT_ACTION_SYSTEM_PROMPT

# Load dataset
def load_prompt_dataset() -> Dataset:
    import json
    data = []
    with open("data/molforge_sft_compact_policy_v4.jsonl", "r") as f:
        for line in f:
            record = json.loads(line)
            prompt_text = compact_action_payload(record)
            data.append({
                "prompt": [
                    {"role": "system", "content": COMPACT_ACTION_SYSTEM_PROMPT},
                    {"role": "user", "content": prompt_text}
                ],
                "record": record
            })
    return Dataset.from_list(data)

dataset = load_prompt_dataset()

# Configure GRPO
training_args = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_generations=NUM_GENERATIONS,
    max_steps=RL_MAX_STEPS,
    logging_steps=1,
    save_steps=25,
    bf16=True,
    report_to="none",
    log_completions=True,
)

# Initialize Trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=molforge_reward_func,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting GRPO Training...")
trainer.train()

print(f"Training complete. Saving adapters to {ADAPTER_SAVE_DIR}")
trainer.save_model(str(ADAPTER_SAVE_DIR))
tokenizer.save_pretrained(str(ADAPTER_SAVE_DIR))